# 🎧 Whisper ATC Fine-Tuning Notebook

**Purpose:** Fine-tune `jlvdoorn/whisper-large-v3-atco2-asr` on your American ATC dataset.

**Base Model:** jlvdoorn/whisper-large-v3-atco2-asr (ATCO2 ASR fine-tuned)
**Base Model WER:** 17.04%

**Author:** Jeffrey

---

## Instructions:
1. **Runtime:** Go to `Runtime > Change runtime type > A100 GPU`
2. **Upload Data:** Upload your `atc_training_dataset_2026-01-15.zip` file
3. **Run All:** Execute cells in order (Ctrl+F9 or Runtime > Run all)

## 📦 Cell 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate librosa soundfile accelerate
!pip install -q tensorboard jiwer
# Install ffmpeg for MP3 support
!apt-get install -qq ffmpeg
print("✅ Dependencies installed!")

## 📤 Cell 2: Upload and Extract Dataset

In [ ]:
import os
import zipfile
from google.colab import files

# Option 1: Upload ZIP file
print("📤 Please upload your dataset ZIP file...")
print("   (atc_training_dataset_2026-01-15.zip)\n")

uploaded = files.upload()

# Extract the uploaded ZIP
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"\n📂 Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('./atc_training_dataset_2026-01-15')
        print("✅ Extraction complete!")
        
        # Remove the ZIP to save space
        os.remove(filename)
        print(f"🗑️ Removed {filename} to save space")

# List extracted contents
print("\n📁 Contents:")
!ls -la ./atc_training_dataset_2026-01-15/

## ⚙️ Cell 3: Configuration

In [ ]:
import os
import torch
import warnings
warnings.filterwarnings('ignore')

# ======== PATHS - UPDATE IF NEEDED ========
DATASET_DIR = "./atc_training_dataset_2026-01-15"
METADATA_FILE = f"{DATASET_DIR}/metadata.jsonl"
OUTPUT_DIR = "./whisper-american-atc-finetuned"

# ======== MODEL ========
MODEL_NAME = "jlvdoorn/whisper-large-v3-atco2-asr"
BASE_MODEL_WER = 17.04  # Base model WER for comparison

# ======== TRAINING HYPERPARAMETERS ========
# Matching original model's training config
LEARNING_RATE = 1e-5
BATCH_SIZE = 16  # Original model used 16
GRADIENT_ACCUMULATION = 1
NUM_EPOCHS = 3
WARMUP_STEPS = 100  # Original model used 100 warmup steps
WEIGHT_DECAY = 0.01

# ======== AUDIO ========
SAMPLE_RATE = 16000

# ======== SPLIT ========
VAL_SPLIT = 0.15
RANDOM_SEED = 42

# ======== GPU CHECK ========
device = "cuda" if torch.cuda.is_available() else "cpu"
print("\n" + "="*60)
print(" 🖥️  HARDWARE CHECK")
print("="*60)
print(f"\n Device: {device}")

# Precision settings
USE_FP16 = False
USE_BF16 = False

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU: {gpu_name}")
    print(f" VRAM: {gpu_memory:.1f} GB")
    
    # Check for bf16 support (Ampere and newer)
    if torch.cuda.is_bf16_supported():
        USE_BF16 = True
        print(" ✅ BF16 supported - using bf16 precision")
    else:
        USE_FP16 = True
        print(" ✅ Using FP16 precision")
    
    if "A100" in gpu_name:
        print(" ✅ A100 detected - optimal for training!")
        BATCH_SIZE = 16
        GRADIENT_ACCUMULATION = 1
    elif "T4" in gpu_name:
        print(" ⚠️ T4 detected - reducing batch size...")
        BATCH_SIZE = 4
        GRADIENT_ACCUMULATION = 4
    elif "V100" in gpu_name:
        print(" ✅ V100 detected")
        BATCH_SIZE = 8
        GRADIENT_ACCUMULATION = 2
    elif gpu_memory < 16:
        print(f" ⚠️ Limited VRAM ({gpu_memory:.1f}GB) - reducing batch size...")
        BATCH_SIZE = 4
        GRADIENT_ACCUMULATION = 4
else:
    print(" ❌ No GPU! Go to Runtime > Change runtime type > GPU")

print(f"\n Precision: {'BF16' if USE_BF16 else 'FP16' if USE_FP16 else 'FP32'}")
print(f" Batch size: {BATCH_SIZE}")
print(f" Gradient accumulation: {GRADIENT_ACCUMULATION}")
print(f" Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print("="*60 + "\n")

# Verify dataset exists
if os.path.exists(DATASET_DIR):
    print(f"✅ Dataset found: {DATASET_DIR}")
    # Check for audio files
    audio_dir = os.path.join(DATASET_DIR, "audio")
    if os.path.exists(audio_dir):
        audio_files = [f for f in os.listdir(audio_dir) if f.endswith(('.mp3', '.wav', '.flac'))]
        print(f"   Audio files: {len(audio_files)}")
    if os.path.exists(METADATA_FILE):
        with open(METADATA_FILE, 'r') as f:
            num_samples = sum(1 for _ in f)
        print(f"   Metadata entries: {num_samples}")
else:
    print(f"❌ Dataset not found: {DATASET_DIR}")
    print("   Please run Cell 2 to upload your dataset.")

## 📚 Cell 4: Import Libraries & Define Functions

In [ ]:
import os
import json
import torch
import numpy as np
import librosa
from pathlib import Path
from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from transformers import pipeline

print("✅ Libraries imported!")

In [ ]:
def load_jsonl_dataset(dataset_dir: str, metadata_file: str) -> Dataset:
    """Load dataset from JSONL metadata format."""
    print(f"📂 Loading dataset from: {dataset_dir}")
    
    samples = []
    with open(metadata_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))
    
    print(f"   Found {len(samples)} samples")
    
    data = {"audio": [], "text": [], "duration": [], "source": [], "audio_path": []}
    processed = 0
    
    for sample in samples:
        audio_path = os.path.join(dataset_dir, sample["audio"])
        if not os.path.exists(audio_path):
            continue
        
        transcript = sample.get("text", "").strip()
        if not transcript:
            continue
        
        try:
            audio_array, _ = librosa.load(audio_path, sr=SAMPLE_RATE)
            data["audio"].append({"array": audio_array, "sampling_rate": SAMPLE_RATE})
            data["text"].append(transcript)
            data["duration"].append(sample.get("duration", len(audio_array) / SAMPLE_RATE))
            data["source"].append(sample.get("source", "unknown"))
            data["audio_path"].append(audio_path)
            processed += 1
            if processed % 50 == 0:
                print(f"   Loaded {processed} files...")
        except Exception as e:
            continue
    
    dataset = Dataset.from_dict(data)
    total_duration = sum(data["duration"])
    
    print(f"\n✅ Loaded {len(dataset)} samples ({total_duration/60:.1f} minutes)")
    return dataset


def prepare_dataset(batch: Dict, processor: WhisperProcessor) -> Dict:
    """Convert audio to Whisper input format."""
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


@dataclass
class WhisperDataCollator:
    """Data collator for Whisper training."""
    processor: Any

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        return batch

print("✅ Functions defined!")

## 📥 Cell 5: Load Dataset

In [ ]:
print("="*60)
print(" STEP 1: LOADING DATASET")
print("="*60 + "\n")

dataset = load_jsonl_dataset(DATASET_DIR, METADATA_FILE)

# Split
splits = dataset.train_test_split(test_size=VAL_SPLIT, seed=RANDOM_SEED, shuffle=True)
train_dataset = splits["train"]
eval_dataset = splits["test"]

# Save for evaluation
eval_texts = eval_dataset["text"]
eval_audio_paths = eval_dataset["audio_path"]

print(f"\n📊 Train set: {len(train_dataset)} samples")
print(f"📊 Eval set: {len(eval_dataset)} samples")

## 🤖 Cell 6: Load Model

In [ ]:
print("="*60)
print(" STEP 2: LOADING MODEL")
print("="*60 + "\n")

print(f"📥 Loading: {MODEL_NAME}")
print("   This may take 2-3 minutes...\n")

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Configure for training - IMPORTANT: disable cache before enabling gradient checkpointing
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# Enable gradient checkpointing with use_reentrant=False to avoid backward pass issues
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

print(f"\n✅ Model loaded!")
print(f"   Parameters: {model.num_parameters() / 1e9:.2f}B")

## 🔄 Cell 7: Preprocess Data

In [ ]:
print("="*60)
print(" STEP 3: PREPROCESSING DATA")
print("="*60 + "\n")

print("🔄 Converting audio to Whisper format...")

columns_to_remove = ["audio", "text", "duration", "source", "audio_path"]

train_processed = train_dataset.map(
    lambda x: prepare_dataset(x, processor),
    remove_columns=columns_to_remove,
    desc="Processing train"
)

eval_processed = eval_dataset.map(
    lambda x: prepare_dataset(x, processor),
    remove_columns=columns_to_remove,
    desc="Processing eval"
)

# Check feature shape (handle both list and array formats)
features = train_processed[0]['input_features']
if isinstance(features, list):
    print(f"\n✅ Done! Feature shape: ({len(features)}, {len(features[0]) if features else 0})")
else:
    print(f"\n✅ Done! Feature shape: {features.shape}")

## 🚀 Cell 8: Train Model

In [ ]:
print("="*60)
print(" STEP 4: TRAINING")
print("="*60 + "\n")

data_collator = WhisperDataCollator(processor=processor)

effective_batch = BATCH_SIZE * GRADIENT_ACCUMULATION
steps_per_epoch = max(1, len(train_dataset) // effective_batch)
total_steps = steps_per_epoch * NUM_EPOCHS

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to=["tensorboard"],
    # Precision settings - auto-detected based on GPU
    fp16=USE_FP16,
    bf16=USE_BF16,
    # Gradient checkpointing with non-reentrant mode
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    dataloader_num_workers=2,
    push_to_hub=False,
    seed=RANDOM_SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=eval_processed,
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)

print(f"📊 Settings:")
print(f"   Model: {MODEL_NAME}")
print(f"   Precision: {'BF16' if USE_BF16 else 'FP16' if USE_FP16 else 'FP32'}")
print(f"   Effective batch: {effective_batch}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Warmup steps: {WARMUP_STEPS}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Total steps: ~{total_steps}")
print(f"\n🚀 Starting training...\n")

train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Final loss: {train_result.training_loss:.4f}")

## 💾 Cell 9: Save Model

In [ ]:
print("="*60)
print(" STEP 5: SAVING MODEL")
print("="*60 + "\n")

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"✅ Model saved to: {OUTPUT_DIR}")
print("\n📁 Files:")
!ls -lh {OUTPUT_DIR}/

## 📊 Cell 10: Evaluate & Test

In [ ]:
print("="*60)
print(" STEP 6: EVALUATION")
print("="*60 + "\n")

wer_metric = evaluate.load("wer")

print("🔍 Loading fine-tuned model for testing...")
finetuned_pipe = pipeline(
    "automatic-speech-recognition",
    model=OUTPUT_DIR,
    device=0,
    chunk_length_s=30
)

num_test = min(10, len(eval_texts))
predictions = []
references = []

print(f"\n📝 Sample Predictions ({num_test} samples):\n")

for i in range(num_test):
    audio, sr = librosa.load(eval_audio_paths[i], sr=SAMPLE_RATE)
    result = finetuned_pipe({"array": audio, "sampling_rate": sr})
    
    pred = result["text"]
    ref = eval_texts[i]
    predictions.append(pred)
    references.append(ref)
    
    print(f"--- Sample {i+1} ---")
    print(f"Ground Truth: {ref}")
    print(f"Prediction:   {pred}\n")

test_wer = 100 * wer_metric.compute(predictions=predictions, references=references)

print("="*60)
print(" 📊 RESULTS")
print("="*60)
print(f"\n   Base Model: {MODEL_NAME}")
print(f"   Base Model WER: {BASE_MODEL_WER:.2f}%")
print(f"   Fine-tuned WER: {test_wer:.2f}%")

if test_wer < BASE_MODEL_WER:
    improvement = BASE_MODEL_WER - test_wer
    print(f"\n   🎉 IMPROVEMENT: {improvement:.2f}% better!")
else:
    print(f"\n   📊 Performance comparable to base model")

## 📥 Cell 11: Download Model

In [ ]:
import shutil
from google.colab import files

print("📦 Creating ZIP file for download...")

# Create ZIP of the model
zip_name = "whisper-american-atc-finetuned"
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)

print(f"✅ Created {zip_name}.zip")
print("\n📥 Starting download...")

files.download(f"{zip_name}.zip")

## 🎯 Done!

Your fine-tuned model has been:
1. ✅ Trained on your American ATC dataset
2. ✅ Saved to Google Drive / Colab
3. ✅ Evaluated on validation samples
4. ✅ Zipped for download

**Next steps:**
- Test on new audio samples
- Upload to Hugging Face Hub (optional)
- Deploy as inference endpoint